# Transformer Fine-Tuning — DistilBERT on SST-2 (Sentiment Analysis)

This notebook demonstrates how to **fine-tune a pre-trained Transformer model** for text classification using **PyTorch** and the **HuggingFace ecosystem**.

**What you will learn:**
- What transfer learning is and why fine-tuning works
- How to load a pre-trained model and tokenizer from HuggingFace
- How tokenization works (input IDs, attention masks, special tokens)
- How to use the HuggingFace `Trainer` API for fine-tuning
- How to run inference on custom sentences

**Model:** `distilbert-base-uncased` — a smaller, faster version of BERT (40% smaller, 60% faster, retains 97% of BERT's performance).

**Dataset:** SST-2 (Stanford Sentiment Treebank) — 67,349 movie review sentences labeled as positive (1) or negative (0).

Loaded from the GLUE benchmark via HuggingFace `datasets`.

## Step 1 — Install Required Packages

We need:
- `transformers`: Pre-trained models and tokenizers from HuggingFace
- `datasets`: Easy-to-use dataset loading
- `evaluate`: Metrics computation (accuracy, F1, etc.)
- `accelerate`: Enables efficient training (required by `Trainer`)
- `torch`: The deep learning framework

In [1]:
!pip install torch transformers datasets evaluate accelerate --quiet

## Step 2 — Imports & Device Setup

In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Model name — DistilBERT is a good balance of speed and performance
MODEL_NAME = "distilbert-base-uncased"

/Users/emirozturk/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## Step 3 — Load & Explore the SST-2 Dataset

SST-2 is part of the GLUE benchmark — a collection of NLP tasks. Each sample has:
- `sentence`: A movie review sentence
- `label`: 0 (negative) or 1 (positive)

Note: The official test set labels are hidden (for leaderboard submission), so we use the **validation** set as our test set.

In [3]:
# Load SST-2 from GLUE benchmark
dataset = load_dataset("glue", "sst2")

print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")

# Show some examples
print("\n--- Sample Sentences ---")
for i in range(5):
    sample = dataset['train'][i]
    sentiment = "Positive" if sample['label'] == 1 else "Negative"
    print(f"  [{sentiment:>8s}] {sample['sentence']}")

# Label distribution
from collections import Counter
label_dist = Counter(dataset['train']['label'])
print(f"\nLabel distribution: Negative={label_dist[0]}, Positive={label_dist[1]}")

Generating test split: 100%|██████████| 1821/1821 [00:00<00:00, 909048.75 examples/s]


Train samples: 67349
Validation samples: 872

--- Sample Sentences ---
  [Negative] hide new secretions from the parental units 
  [Negative] contains no wit , only labored gags 
  [Positive] that loves its characters and communicates something rather beautiful about human nature 
  [Negative] remains utterly satisfied to remain the same throughout 
  [Negative] on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 

Label distribution: Negative=29780, Positive=37569


## Step 4 — Load the Pre-Trained Tokenizer

A **tokenizer** converts text into the format the model expects:
1. Splits text into **subword tokens** (not just words — "unbelievable" might become "un", "##believ", "##able")
2. Converts tokens to **input IDs** (integer indices)
3. Adds **special tokens**: `[CLS]` at the start, `[SEP]` at the end
4. Creates an **attention mask**: 1 for real tokens, 0 for padding

DistilBERT uses the **WordPiece** tokenizer (same as BERT).

In [4]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Demonstrate how tokenization works
example_text = "This movie was absolutely fantastic!"
encoded = tokenizer(example_text, return_tensors="pt")

print(f"Original text: '{example_text}'")
print(f"\nTokens: {tokenizer.tokenize(example_text)}")
print(f"Input IDs: {encoded['input_ids'].tolist()}")
print(f"Attention mask: {encoded['attention_mask'].tolist()}")

# Decode back
print(f"Decoded: {tokenizer.decode(encoded['input_ids'][0])}")

# Show special tokens
print(f"\nSpecial tokens:")
print(f"  [CLS] token ID: {tokenizer.cls_token_id}")
print(f"  [SEP] token ID: {tokenizer.sep_token_id}")
print(f"  [PAD] token ID: {tokenizer.pad_token_id}")
print(f"  Vocabulary size: {tokenizer.vocab_size}")

Original text: 'This movie was absolutely fantastic!'

Tokens: ['this', 'movie', 'was', 'absolutely', 'fantastic', '!']
Input IDs: [[101, 2023, 3185, 2001, 7078, 10392, 999, 102]]
Attention mask: [[1, 1, 1, 1, 1, 1, 1, 1]]
Decoded: [CLS] this movie was absolutely fantastic! [SEP]

Special tokens:
  [CLS] token ID: 101
  [SEP] token ID: 102
  [PAD] token ID: 0
  Vocabulary size: 30522


## Step 5 — Tokenize the Dataset

We apply the tokenizer to the entire dataset using `.map()`. This is efficient because it processes data in batches.

**Key parameters:**
- `truncation=True`: Cut sequences longer than `max_length` (model has a max of 512 tokens)
- `padding="max_length"`: Pad all sequences to the same length (required for batching)
- `max_length=128`: Maximum sequence length (128 is enough for single sentences, saves memory)

In [5]:
def tokenize_function(examples):
    """Tokenize a batch of examples."""
    return tokenizer(
        examples['sentence'],
        truncation=True,        # Truncate to max_length
        padding='max_length',   # Pad to max_length
        max_length=128          # Maximum token length
    )

# Apply tokenization to the entire dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# The tokenizer adds 'input_ids' and 'attention_mask' columns
print(f"Columns after tokenization: {tokenized_dataset['train'].column_names}")

# Look at a tokenized sample
sample = tokenized_dataset['train'][0]
print(f"\nTokenized sample:")
print(f"  Sentence: {dataset['train'][0]['sentence']}")
print(f"  Input IDs (first 20): {sample['input_ids'][:20]}")
print(f"  Attention mask (first 20): {sample['attention_mask'][:20]}")
print(f"  Label: {sample['label']}")

Map: 100%|██████████| 1821/1821 [00:00<00:00, 32681.21 examples/s]

Columns after tokenization: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']

Tokenized sample:
  Sentence: hide new secretions from the parental units 
  Input IDs (first 20): [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Attention mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Label: 0


## Step 6 — Load the Pre-Trained Model

We load `DistilBertForSequenceClassification` — this is DistilBERT with a **classification head** on top.

The model architecture:
```
Input IDs + Attention Mask
  → DistilBERT Encoder (6 Transformer layers, pre-trained on BookCorpus + Wikipedia)
  → [CLS] token output (768-dim vector summarizing the whole sentence)
  → Linear(768, 2) — classification into positive/negative
```

**Transfer Learning:** The DistilBERT encoder was pre-trained on millions of sentences via **masked language modeling** (MLM). It already "understands" English grammar, semantics, and common sense. We only need to fine-tune it for our specific task (sentiment classification).

The `num_labels=2` warning is expected — we're initializing a new classification head that hasn't been trained yet.

In [6]:
# Load model with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # Binary classification: positive/negative
)

print(f"Model type: {type(model).__name__}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 13763.10it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model type: DistilBertForSequenceClassification
Total parameters: 66,955,010
Trainable parameters: 66,955,010


## Step 7 — Define Training Arguments

The `TrainingArguments` class configures everything about the training process.

**Key parameters explained:**
- `output_dir`: Where to save checkpoints and logs
- `num_train_epochs`: How many times to iterate over the training data
- `per_device_train_batch_size`: Batch size per GPU/CPU
- `learning_rate`: Lower than typical (2e-5) — we don't want to destroy pre-trained knowledge
- `weight_decay`: L2 regularization to prevent overfitting
- `eval_strategy="epoch"`: Run evaluation at the end of each epoch
- `save_strategy="epoch"`: Save a checkpoint after each epoch
- `load_best_model_at_end`: After training, load the checkpoint with the best eval metric

In [7]:
training_args = TrainingArguments(
    output_dir="./results",               # Checkpoint directory
    num_train_epochs=3,                    # 3 epochs is typical for fine-tuning
    per_device_train_batch_size=16,        # Batch size (reduce if out of memory)
    per_device_eval_batch_size=32,         # Larger eval batch (no gradients needed)
    learning_rate=2e-5,                    # Low LR — preserve pre-trained knowledge
    weight_decay=0.01,                     # L2 regularization
    eval_strategy="epoch",                 # Evaluate at end of each epoch
    save_strategy="epoch",                 # Save checkpoint each epoch
    load_best_model_at_end=True,           # Keep the best model
    metric_for_best_model="accuracy",      # Best = highest accuracy
    logging_steps=100,                     # Log every 100 steps
    report_to="none",                      # Don't report to wandb/tensorboard
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Weight decay: {training_args.weight_decay}")

Training configuration:
  Epochs: 3
  Batch size: 16
  Learning rate: 2e-05
  Weight decay: 0.01


## Step 8 — Define Evaluation Metric

We define a function that computes **accuracy** during evaluation. The `Trainer` calls this after each evaluation step.

In [8]:
# Load the accuracy metric
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """
    Compute accuracy from model predictions.
    
    Args:
        eval_pred: tuple of (logits, labels)
            logits: numpy array of shape (num_samples, num_classes)
            labels: numpy array of shape (num_samples,)
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)  # Get class with highest logit
    return accuracy_metric.compute(predictions=predictions, references=labels)

print("Evaluation metric: Accuracy")

Evaluation metric: Accuracy


## Step 9 — Create the Trainer & Fine-Tune

The `Trainer` class handles the entire training loop:
- Batching, shuffling, gradient accumulation
- Mixed precision (FP16) if available
- Gradient clipping
- Learning rate scheduling (linear warmup + decay)
- Checkpoint saving and loading
- Evaluation at specified intervals

This is the "batteries-included" approach — much less code than writing a training loop from scratch.

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
)

print("Trainer created. Starting fine-tuning...")
print(f"Training on {len(tokenized_dataset['train'])} samples")
print(f"Evaluating on {len(tokenized_dataset['validation'])} samples")

# Fine-tune!
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Total training time: {train_result.metrics['train_runtime']:.1f} seconds")
print(f"Training loss: {train_result.metrics['train_loss']:.4f}")

Trainer created. Starting fine-tuning...
Training on 67349 samples
Evaluating on 872 samples


/Users/emirozturk/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.174151,0.293353,0.905963
2,0.117092,0.379293,0.907110
3,0.085927,0.416798,0.903670


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.09it/s]
/Users/emirozturk/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.35it/s]
/Users/emirozturk/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.59it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Training complete!
Total training time: 1634.4 seconds
Training loss: 0.1471


## Step 10 — Evaluate on Validation Set

Let's get the final evaluation results on the validation set using the best checkpoint.

In [10]:
# Remove the notebook progress callback that causes issues when evaluate()
# is called in a separate cell from train()
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

eval_results = trainer.evaluate()

print("=== Evaluation Results ===")
print(f"Accuracy: {eval_results['eval_accuracy']:.4f} ({eval_results['eval_accuracy']*100:.2f}%)")
print(f"Loss: {eval_results['eval_loss']:.4f}")

/Users/emirozturk/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: on_train_begin must be called before on_evaluate

## Step 11 — Inference on Custom Sentences

Now let's use our fine-tuned model to classify custom sentences. This shows the practical end-to-end pipeline:
1. Tokenize the input text
2. Feed through the model
3. Apply softmax to get probabilities
4. Take the argmax for the predicted class

In [ ]:
def predict_sentiment(text, model, tokenizer):
    """Predict sentiment for a single sentence."""
    model.eval()
    
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(model.device)
    
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_class].item()
    
    sentiment = "Positive" if pred_class == 1 else "Negative"
    return sentiment, confidence

# Test on custom sentences
test_sentences = [
    "This is the best movie I've ever seen! Absolutely brilliant.",
    "A complete waste of time. Terrible acting and awful plot.",
    "The film was okay, nothing special but watchable.",
    "An incredible journey with stunning visuals and a powerful story.",
    "I was bored throughout the entire movie. Very disappointing.",
    "Not bad, but I expected more from the director.",
    "A masterpiece of modern cinema.",
    "The worst sequel ever made."
]

print("=== Custom Sentence Predictions ===")
for sentence in test_sentences:
    sentiment, confidence = predict_sentiment(sentence, model, tokenizer)
    print(f"\n\"{sentence}\"")
    print(f"  → {sentiment} (confidence: {confidence:.4f})")

## Step 12 — Understanding What Happened Under the Hood

### What is Transfer Learning?

Instead of training a model from scratch (which requires massive data and compute), we:
1. **Start with a pre-trained model** that already understands language
2. **Fine-tune** it on our specific task with a small dataset

### Why Does Fine-Tuning Work?

DistilBERT was pre-trained on:
- **BookCorpus**: 11,038 books (~800M words)
- **English Wikipedia**: ~2.5B words

Through **Masked Language Modeling** (predicting masked words) and **distillation** from BERT, it learned:
- Grammar and syntax
- Word meanings and relationships
- Common sense knowledge
- Sentiment-carrying phrases (even before fine-tuning!)

Fine-tuning only needs to:
- Slightly adjust the existing representations for sentiment
- Train the new classification head (768 → 2)

### Key Differences from Training from Scratch

| Aspect | From Scratch | Fine-Tuning |
|--------|-------------|-------------|
| **Data needed** | Millions of samples | Thousands |
| **Training time** | Days/weeks | Minutes/hours |
| **Learning rate** | Higher (1e-3) | Much lower (2e-5) |
| **Epochs** | 50-100+ | 2-5 |
| **Performance** | Limited by data | State-of-the-art |

### The Transformer Architecture (Simplified)

```
Input: "This movie is great"
  → Tokenize: [CLS] this movie is great [SEP]
  → Embed: each token gets a 768-dim vector (token embedding + position embedding)
  → 6× Transformer Layers:
      → Multi-Head Self-Attention (each token attends to all others)
      → Feed-Forward Network
      → Layer Normalization + Residual Connections
  → [CLS] output: 768-dim vector summarizing the whole sentence
  → Classification Head: Linear(768, 2)
  → Softmax → [0.02, 0.98] → Positive!
```

The **self-attention** mechanism is what makes Transformers powerful — every token can directly attend to every other token, giving them global context (unlike RNNs which process sequentially).

## Summary

In this notebook we:
1. **Loaded** the SST-2 sentiment dataset (67k sentences)
2. **Loaded** a pre-trained DistilBERT tokenizer and model from HuggingFace
3. **Tokenized** the dataset (text → input IDs + attention masks)
4. **Configured** training with `TrainingArguments` (3 epochs, lr=2e-5)
5. **Fine-tuned** with the `Trainer` API
6. **Evaluated** on the validation set (typically ~91-92% accuracy)
7. **Ran inference** on custom sentences
8. **Explained** transfer learning and the Transformer architecture

**Key takeaways:**
- Fine-tuning pre-trained Transformers gives state-of-the-art results with minimal effort
- The HuggingFace ecosystem (`transformers` + `datasets` + `Trainer`) makes this very accessible
- Low learning rates are critical — you don't want to destroy pre-trained knowledge
- DistilBERT is a great starting point — fast enough for experimentation, powerful enough for production

**Next steps to try:**
- Try `bert-base-uncased` or `roberta-base` for potentially higher accuracy
- Use `Trainer` with early stopping (`EarlyStoppingCallback`)
- Try on other GLUE tasks: MRPC (paraphrase), QNLI (QA), CoLA (grammar)
- Export the model with `model.save_pretrained()` for deployment